# Fourier Transform & Reciprocal Lattice Explorer

Interactively explore how real-space atom positions and unit-cell geometry
affect reciprocal-space diffraction patterns.

In [15]:
import numpy as np
import plotly.graph_objects as go
from ipywidgets import (
    HBox, VBox, Button, FloatSlider, Layout, Label, Output
)
from IPython.display import display

# ============================================================
#                INITIALIZATION FLAGS
# ============================================================
initializing = True       # prevents Voilà freeze during load
suspend_callbacks = False # used for reset operations

# ============================================================
#                INITIAL PARAMETERS
# ============================================================
a_init, b_init = 10.0, 10.0
gamma_init = 90.0
nx_real, ny_real = 3, 3
nx_recip, ny_recip = 20, 20
sigma = 0.2
atoms_frac_init = np.array([[0.0, 0.0], [0.5, 0.5]])
num_atoms = len(atoms_frac_init)
real_grid_size = 512
qmax_plot = np.pi

atoms_frac = atoms_frac_init.copy()
a, b, gamma = a_init, b_init, gamma_init
atom_mask = np.ones(num_atoms, dtype=bool)

# ============================================================
#           LATTICE GEOMETRY FUNCTIONS
# ============================================================
def unit_cell_vectors(a_val, b_val, gamma_deg):
    g = np.deg2rad(gamma_deg)
    a_vec = np.array([a_val, 0.0])
    b_vec = np.array([b_val * np.cos(g), b_val * np.sin(g)])
    return a_vec, b_vec

def frac_to_cart(frac, a_val, b_val, gamma_deg):
    a_vec, b_vec = unit_cell_vectors(a_val, b_val, gamma_deg)
    return np.outer(frac[:, 0], a_vec) + np.outer(frac[:, 1], b_vec)

def tile_unit_cell(atoms_cart_local, nx_val, ny_val, a_val, b_val, gamma_deg, center=True):
    a_vec, b_vec = unit_cell_vectors(a_val, b_val, gamma_deg)
    pts = []
    for i in range(nx_val):
        for j in range(ny_val):
            pts.append(atoms_cart_local + i*a_vec + j*b_vec)
    pts = np.vstack(pts) if len(pts) else np.zeros((0,2))

    if center:
        pts -= (1.5*a_vec + 1.5*b_vec)

    return pts

# ============================================================
#          REAL SPACE DENSITY + 2D FFT
# ============================================================
def build_density(points, grid_size=real_grid_size):
    x = np.linspace(-25, 25, grid_size)
    y = np.linspace(-25, 25, grid_size)
    X, Y = np.meshgrid(x, y)
    rho = np.zeros_like(X)

    for px, py in points:
        rho += np.exp(-((X - px)**2 + (Y - py)**2)/(2*sigma**2))

    return rho, x, y

def compute_fft_q(rho, x, y):
    ny, nx = rho.shape
    dx = x[1] - x[0]
    dy = y[1] - y[0]

    F = np.fft.fftshift(np.fft.fft2(rho))
    intensity = np.abs(F)**2

    fx = np.fft.fftshift(np.fft.fftfreq(nx, d=dx))
    fy = np.fft.fftshift(np.fft.fftfreq(ny, d=dy))
    qx = 2*np.pi*fx
    qy = 2*np.pi*fy
    return intensity, qx, qy

def crop_q_window(intensity, qx, qy, qmax):
    mask_x = np.where(np.abs(qx) <= qmax)[0]
    mask_y = np.where(np.abs(qy) <= qmax)[0]

    if mask_x.size == 0 or mask_y.size == 0:
        return intensity, qx, qy

    ix0, ix1 = mask_x[0], mask_x[-1]
    iy0, iy1 = mask_y[0], mask_y[-1]

    return (
        intensity[iy0:iy1+1, ix0:ix1+1],
        qx[ix0:ix1+1],
        qy[iy0:iy1+1]
    )

# ============================================================
#                UNIT CELL OVERLAY
# ============================================================
def draw_unit_cell_overlay(fig, nx_val, ny_val, a_vec, b_vec, opacity=0.5):
    fig.layout.shapes = []
    cshift = 1.5*a_vec + 1.5*b_vec

    for i in range(nx_val):
        for j in range(ny_val):
            o = i*a_vec + j*b_vec - cshift
            c0 = o
            c1 = o + a_vec
            c2 = o + a_vec + b_vec
            c3 = o + b_vec

            fig.add_shape(
                type="path",
                path=f"M {c0[0]},{c0[1]} L {c1[0]},{c1[1]} "
                     f"L {c2[0]},{c2[1]} L {c3[0]},{c3[1]} Z",
                line=dict(color=f"rgba(255,255,255,{opacity})", width=2),
                fillcolor="rgba(255,255,255,0)"
            )

# ============================================================
#        OUTPUT WIDGET FIGURES (Binder/Voila safe)
# ============================================================
real_fig = go.Figure()
recip_fig = go.Figure()

real_fig.add_trace(go.Heatmap(z=[[0]], colorscale="Viridis", showscale=False))
recip_fig.add_trace(go.Heatmap(z=[[0]], colorscale="Viridis", showscale=False))

# Make the figures 50% larger
real_fig.update_layout(width=500, height=500)
recip_fig.update_layout(width=500, height=500)

real_out = Output()
recip_out = Output()

with real_out:
    display(real_fig)

with recip_out:
    display(recip_fig)

# ============================================================
#              UPDATE FUNCTION (Voila-safe)
# ============================================================
def update_figures():
    global atoms_cart, rho, x, y

    # ---------------- REAL SPACE ----------------
    atoms_cart = frac_to_cart(atoms_frac, a, b, gamma)
    active_atoms = atoms_cart[atom_mask]

    pts = tile_unit_cell(active_atoms, nx_real, ny_real, a, b, gamma, center=True)
    rho, x, y = build_density(pts)

    real_fig.data[0].z = rho
    real_fig.data[0].x = x
    real_fig.data[0].y = y

    draw_unit_cell_overlay(real_fig, nx_real, ny_real, *unit_cell_vectors(a, b, gamma))

    with real_out:
        real_out.clear_output(wait=True)
        display(real_fig)

    # Keep real-space square
    real_fig.update_yaxes(scaleanchor="x", scaleratio=1)
    real_fig.update_xaxes(constrain="domain")

    # ---------------- RECIPROCAL SPACE ----------------
    fft_full, qx_full, qy_full = compute_fft_q(rho, x, y)
    fft_crop, qx_axis, qy_axis = crop_q_window(fft_full, qx_full, qy_full, qmax_plot)

    recip_fig.data[0].z = fft_crop
    recip_fig.data[0].x = qx_axis
    recip_fig.data[0].y = qy_axis

    recip_fig.update_xaxes(range=[-qmax_plot, qmax_plot], title="qₓ (Å⁻¹)")
    recip_fig.update_yaxes(range=[-qmax_plot, qmax_plot], title="qᵧ (Å⁻¹)")

    with recip_out:
        recip_out.clear_output(wait=True)
        display(recip_fig)

    # Keep reciprocal square
    recip_fig.update_yaxes(scaleanchor="x", scaleratio=1)
    recip_fig.update_xaxes(constrain="domain")

# ============================================================
#              FRACTIONAL SLIDERS (with guards)
# ============================================================
sliders = []
slider_boxes = []

for i in range(num_atoms):
    fx = FloatSlider(description=f"Atom {i} fₓ", min=0, max=1, step=0.001,
                     value=float(atoms_frac[i,0]), layout=Layout(width="420px"))
    fy = FloatSlider(description=f"Atom {i} fᵧ", min=0, max=1, step=0.001,
                     value=float(atoms_frac[i,1]), layout=Layout(width="420px"))

    sliders.append((fx, fy))

    def make_callback(idx):
        def cb(change):
            global initializing, suspend_callbacks
            if initializing or suspend_callbacks:
                return
            atoms_frac[idx,0] = sliders[idx][0].value
            atoms_frac[idx,1] = sliders[idx][1].value
            update_figures()
        return cb

    fx.observe(make_callback(i), "value")
    fy.observe(make_callback(i), "value")

    slider_boxes.append(VBox([fx, fy]))

# ============================================================
#                    UNIT CELL SLIDERS
# ============================================================
a_slider = FloatSlider(description="a (Å)", min=1, max=30, step=0.1, value=a_init)
b_slider = FloatSlider(description="b (Å)", min=1, max=30, step=0.1, value=b_init)
gamma_slider = FloatSlider(description="Gamma (°)", min=30, max=150, step=0.5, value=gamma_init)

def on_cell_change(change):
    global a, b, gamma
    if initializing:
        return
    a = a_slider.value
    b = b_slider.value
    gamma = gamma_slider.value
    update_figures()

a_slider.observe(on_cell_change, "value")
b_slider.observe(on_cell_change, "value")
gamma_slider.observe(on_cell_change, "value")

# ============================================================
#                    BUTTONS
# ============================================================
reset_btn = Button(description="Reset", button_style="primary")
def on_reset(btn):
    global a, b, gamma, atoms_frac, atom_mask, suspend_callbacks

    suspend_callbacks = True
    a, b, gamma = a_init, b_init, gamma_init
    atoms_frac[:] = atoms_frac_init
    atom_mask[:] = True

    a_slider.value = a
    b_slider.value = b
    gamma_slider.value = gamma

    for i in range(num_atoms):
        sliders[i][0].value = atoms_frac[i,0]
        sliders[i][1].value = atoms_frac[i,1]

    suspend_callbacks = False
    update_figures()

reset_btn.on_click(on_reset)

toggle_btn = Button(description="Toggle Atom 1")
def on_toggle(btn):
    if initializing:
        return
    atom_mask[1] = not atom_mask[1]
    update_figures()

toggle_btn.on_click(on_toggle)

# ============================================================
#                       UI LAYOUT
# ============================================================
controls = VBox([
    Label("Unit Cell Parameters:"), 
    a_slider, b_slider, gamma_slider,
    HBox([reset_btn, toggle_btn])
])

figures_box = HBox([
    real_out,
    recip_out
])

ui = VBox([
    figures_box,
    HBox([
        controls,
        VBox([Label("Atom fractional sliders:")] + slider_boxes)
    ])
])

display(ui)

# ============================================================
#     END OF NOTEBOOK — ENABLE CALLBACKS + FIRST UPDATE
# ============================================================
initializing = False
update_figures()   # show plots immediately
